In [ ]:
# Importar pacotes
import nltk
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import movie_reviews
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import mark_negation, extract_unigram_feats

# Download subjectivity
nltk.download('movie_reviews')

# Carregar docs
n_instances = 1000
pos_docs = [(list(movie_reviews.words(fileid)), 'pos')
            for fileid in movie_reviews.fileids('pos')[:n_instances]]
neg_docs = [(list(movie_reviews.words(fileid)), 'neg')
            for fileid in movie_reviews.fileids('neg')[:n_instances]]

# Definir bases de treino e teste 80/20
train_pos_docs = pos_docs[:1000]
test_pos_docs = pos_docs[800:1000]
train_neg_docs = neg_docs[:800]
test_neg_docs = neg_docs[800:1000]
training_docs = train_pos_docs+train_neg_docs
testing_docs = test_pos_docs+test_neg_docs

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\franc\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [7]:
# SEM TRATAMENTO DE NEGAÇÃO

sentim_analyzer = SentimentAnalyzer()

# Seleção de features
all_words= sentim_analyzer.all_words(training_docs)
unigrams_feats  = sentim_analyzer.unigram_word_feats(all_words, min_freq=4)
sentim_analyzer.add_feat_extractor(extract_unigram_feats, unigrams=unigrams_feats)

# Parametrizar datasets
training_set = sentim_analyzer.apply_features(training_docs)
test_set  = sentim_analyzer.apply_features(testing_docs)

# Treinar modelos
trainer = NaiveBayesClassifier.train
classifier = sentim_analyzer.train(trainer, training_set)
results = sentim_analyzer.evaluate(test_set)

for key,value in sorted(sentim_analyzer.evaluate(test_set).items()):
     print('{0}: {1}'.format(key, value))

Training classifier
Evaluating NaiveBayesClassifier results...
Evaluating NaiveBayesClassifier results...
Accuracy: 0.805
F-measure [neg]: 0.8115942028985507
F-measure [pos]: 0.7979274611398963
Precision [neg]: 0.7850467289719626
Precision [pos]: 0.8279569892473119
Recall [neg]: 0.84
Recall [pos]: 0.77


In [8]:
# COM TRATAMENTO DE NEGAÇÃO

sentim_analyzer_neg = SentimentAnalyzer()

# Marca negação de termos
training_neg = [(mark_negation(doc), label) for doc, label in training_docs]
testing_neg  = [(mark_negation(doc), label) for doc, label in testing_docs]
all_words_neg = sentim_analyzer_neg.all_words(training_neg)

# Seleção de features
unigrams_feats_neg = sentim_analyzer_neg.unigram_word_feats(all_words_neg, min_freq=4)
sentim_analyzer_neg.add_feat_extractor(extract_unigram_feats, unigrams=unigrams_feats_neg)

# Parametrizar datasets
training_set_neg = sentim_analyzer_neg.apply_features(training_neg)
test_set_neg     = sentim_analyzer_neg.apply_features(testing_neg)

# Treinar modelos
trainer_neg = NaiveBayesClassifier.train
classifier_neg = sentim_analyzer_neg.train(trainer_neg, training_set_neg)
results_neg = sentim_analyzer_neg.evaluate(test_set_neg)

for key,value in sorted(sentim_analyzer_neg.evaluate(test_set_neg).items()):
     print('{0}: {1}'.format(key, value))

Training classifier
Evaluating NaiveBayesClassifier results...
Evaluating NaiveBayesClassifier results...
Accuracy: 0.805
F-measure [neg]: 0.8097560975609756
F-measure [pos]: 0.8
Precision [neg]: 0.7904761904761904
Precision [pos]: 0.8210526315789474
Recall [neg]: 0.83
Recall [pos]: 0.78


In [ ]:
# COMPARAÇÃO
metrics = ["Accuracy",
           "F-measure [pos]", "F-measure [neg]",
           "Precision [pos]", "Precision [neg]",
           "Recall [pos]",    "Recall [neg]"]

print(f"\n{'Métrica':<25} {'Baseline':>10} {'+ Negação':>10} {'Δ':>8}")
print("-" * 58)
for m in metrics:
    b = results.get(m, 0)
    n = results_neg.get(m, 0)
    delta = n - b
    sinal = "+" if delta >= 0 else ""
    print(f"  {m:<23} {b:>10.4f} {n:>10.4f} {sinal}{delta:>7.4f}")


Métrica                     Baseline  + Negação        Δ
----------------------------------------------------------
  Accuracy                    0.8050     0.8050 + 0.0000
  F-measure [pos]             0.7979     0.8000 + 0.0021
  F-measure [neg]             0.8116     0.8098 -0.0018
  Precision [pos]             0.8280     0.8211 -0.0069
  Precision [neg]             0.7850     0.7905 + 0.0054
  Recall [pos]                0.7700     0.7800 + 0.0100
  Recall [neg]                0.8400     0.8300 -0.0100
